# Primera entrega | Entendimiento de datos de Nueva York

**Corte:** 30 de septiembre de 2026. **Estado:** análisis reproducible de extractos de hasta 10 000 registros por fuente, no estimaciones del universo. Los metadatos y el manifiesto registran hora, consulta y SHA-256. Ejecute con PySpark 4.x en una sesión disponible; este cuaderno no presupone características de clúster.

Los cuatro conjuntos enlazados son arrestos, pobreza, vehículos implicados en colisiones y SAT 2012. El PDF SAT 2024 es fuente complementaria contextual. No se unen personas, arrestos, vehículos y escuelas a nivel individual. Las tablas de los extractos no sustituyen estadísticas oficiales de toda la ciudad.

In [2]:
from pathlib import Path
import json
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName('NYC_primera_entrega').getOrCreate()
base = Path('datos') if Path('datos/manifest.json').exists() else Path('entregables/datos')
manifest = {x['fuente']: x for x in json.loads((base/'manifest.json').read_text(encoding='utf8'))}
for name,m in manifest.items():
    print(name, 'filas extracto:',m['filas_extracto'],'recuento API en corte:',m['total_api'][0]['count'], 'SHA256:',m['sha256'])
dfs = {n:spark.read.json(str(base/(n+'.jsonl'))) for n in manifest}


arrestos filas extracto: 10000 recuento API en corte: 141870 SHA256: e132b52fa5fea999558199ab02cd3508215ce17528e27939790e62724a9e7bdd
pobreza filas extracto: 3000 recuento API en corte: 68273 SHA256: d8daf24ec29885548b83a5f07762b878838b1bd6bba25dd206f1ce396cdb4ae0
vehiculos filas extracto: 3000 recuento API en corte: 4551002 SHA256: 9ca61bf270aefe095daf05ac78fabedb7e80e33774e34a1d2b059c5c044e180e
educacion filas extracto: 478 recuento API en corte: 478 SHA256: 709f635962084694f219ca99c3c284ac3c43d16aada0f79a5f2e6abebf66e16e


## 1. Estructura y unidad de análisis
Inspeccione el esquema y un registro por fuente. El archivo de vehículos tiene una fila por vehículo, no necesariamente una por colisión. Pobreza registra personas y factores de expansión; SAT 2012, escuelas.

In [4]:
for n,df in dfs.items():
    print('\n',n,'columnas',len(df.columns),'filas del extracto',df.count())
    df.printSchema()
    df.show(2,truncate=40)



 arrestos columnas 19 filas del extracto 10000
root
 |-- age_group: string (nullable = true)
 |-- arrest_boro: string (nullable = true)
 |-- arrest_date: string (nullable = true)
 |-- arrest_key: string (nullable = true)
 |-- arrest_precinct: string (nullable = true)
 |-- geocoded_column: struct (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)
 |-- jurisdiction_code: string (nullable = true)
 |-- ky_cd: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- law_cat_cd: string (nullable = true)
 |-- law_code: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- ofns_desc: string (nullable = true)
 |-- pd_cd: string (nullable = true)
 |-- pd_desc: string (nullable = true)
 |-- perp_race: string (nullable = true)
 |-- perp_sex: string (nullable = true)
 |-- x_coord_cd: string (nullable = true)
 |-- y_coord_cd: string (nullable = true)

+---------+

## 2. Cobertura temporal y 3. Integridad de claves

In [6]:
date_cols={'arrestos':'arrest_date','vehiculos':'crash_date'}
for n,c in date_cols.items():
    dfs[n].select(F.min(c).alias('min'),F.max(c).alias('max'),F.count(F.when(F.col(c).isNull(),1)).alias('nulos')).show()
for n,key in {'arrestos':'arrest_key','pobreza':'serialno','vehiculos':'unique_id','educacion':'dbn'}.items():
    df=dfs[n]
    df.agg(F.count('*').alias('filas'),F.countDistinct(key).alias('claves_distintas'),F.count(F.when(F.col(key).isNull(),1)).alias('claves_nulas')).show()
print('Nota: serialno identifica vivienda; la clave persona combina serialno y sporder.')
dfs['pobreza'].select(F.countDistinct(F.struct('serialno','sporder')).alias('personas_distintas')).show()


+--------------------+--------------------+-----+
|                 min|                 max|nulos|
+--------------------+--------------------+-----+
|2026-01-01T00:00:...|2026-01-13T00:00:...|    0|
+--------------------+--------------------+-----+

+--------------------+--------------------+-----+
|                 min|                 max|nulos|
+--------------------+--------------------+-----+
|2012-07-01T00:00:...|2017-01-03T00:00:...|    0|
+--------------------+--------------------+-----+

+-----+----------------+------------+
|filas|claves_distintas|claves_nulas|
+-----+----------------+------------+
|10000|           10000|           0|
+-----+----------------+------------+

+-----+----------------+------------+
|filas|claves_distintas|claves_nulas|
+-----+----------------+------------+
| 3000|            1178|           0|
+-----+----------------+------------+

+-----+----------------+------------+
|filas|claves_distintas|claves_nulas|
+-----+----------------+------------+
| 

## 4. Completitud y 5. Categorías principales

In [8]:
for n,df in dfs.items():
    expr=[F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast('string'))==''),1)).alias(c) for c in df.columns]
    print(n,'nulos o vacíos por campo')
    df.agg(*expr).show(truncate=False)
for n,c in [('arrestos','arrest_boro'),('vehiculos','vehicle_type'),('vehiculos','contributing_factor_1'),('educacion','school_name')]:
    print(n,c)
    dfs[n].groupBy(c).count().orderBy(F.desc('count')).show(10,truncate=45)


arrestos nulos o vacíos por campo
+---------+-----------+-----------+----------+---------------+---------------+-----------------+-----+--------+----------+--------+---------+---------+-----+-------+---------+--------+----------+----------+
|age_group|arrest_boro|arrest_date|arrest_key|arrest_precinct|geocoded_column|jurisdiction_code|ky_cd|latitude|law_cat_cd|law_code|longitude|ofns_desc|pd_cd|pd_desc|perp_race|perp_sex|x_coord_cd|y_coord_cd|
+---------+-----------+-----------+----------+---------------+---------------+-----------------+-----+--------+----------+--------+---------+---------+-----+-------+---------+--------+----------+----------+
|0        |0          |0          |0         |0              |0              |0                |1    |0       |12        |0       |0        |0        |0    |0      |0        |0       |0         |0         |
+---------+-----------+-----------+----------+---------------+---------------+-----------------+-----+--------+----------+--------+-------

## 6. Geografía y 7. Distribución temporal

In [10]:
dfs['arrestos'].groupBy('arrest_boro').count().orderBy(F.desc('count')).show()
for n,c in date_cols.items():
    d=dfs[n].withColumn('fecha',F.to_date(c)).withColumn('mes',F.date_format('fecha','yyyy-MM'))
    print(n)
    d.groupBy('mes').count().orderBy('mes').show(24)


+-----------+-----+
|arrest_boro|count|
+-----------+-----+
|          K| 3092|
|          B| 2335|
|          M| 2199|
|          Q| 2005|
|          S|  369|
+-----------+-----+

arrestos
+-------+-----+
|    mes|count|
+-------+-----+
|2026-01|10000|
+-------+-----+

vehiculos
+-------+-----+
|    mes|count|
+-------+-----+
|2012-07|   58|
|2012-08|   58|
|2012-09|   66|
|2012-10|   50|
|2012-11|   52|
|2012-12|   72|
|2013-01|   51|
|2013-02|   44|
|2013-03|   55|
|2013-04|   54|
|2013-05|   56|
|2013-06|   52|
|2013-07|   54|
|2013-08|   52|
|2013-09|   64|
|2013-10|   56|
|2013-11|   58|
|2013-12|   68|
|2014-01|   35|
|2014-02|   49|
|2014-03|   55|
|2014-04|   51|
|2014-05|   61|
|2014-06|   70|
+-------+-----+
only showing top 24 rows


## 8. Unidades y duplicación de colisiones
El recuento de `collision_id` es un indicador de colisiones *representadas en el extracto*. No equivale al total de accidentes de Nueva York ni permite medir lesionados porque este archivo no trae ese desenlace.

In [12]:
v=dfs['vehiculos']
v.agg(F.count('*').alias('filas_vehiculos'),F.countDistinct('collision_id').alias('colisiones_representadas'),F.count(F.when(F.col('collision_id').isNull(),1)).alias('id_colision_nulo')).show()
v.groupBy('collision_id').count().groupBy('count').count().orderBy('count').show(10)


+---------------+------------------------+----------------+
|filas_vehiculos|colisiones_representadas|id_colision_nulo|
+---------------+------------------------+----------------+
|           3000|                    2999|               0|
+---------------+------------------------+----------------+

+-----+-----+
|count|count|
+-----+-----+
|    1| 2998|
|    2|    1|
+-----+-----+



## 9. Pobreza: distribución, ponderación y valores válidos

In [14]:
p=dfs['pobreza']
p.groupBy('boro','nycgov_pov_stat').count().orderBy('boro','nycgov_pov_stat').show(30)
p.select('pwgtp','nycgov_pov_stat','nycgov_income','nycgov_threshold').describe().show()
print('No interpretar códigos de estado de pobreza sin diccionario oficial. Tasas poblacionales requieren pesos pwgtp y definición documentada.')


+----+---------------+-----+
|boro|nycgov_pov_stat|count|
+----+---------------+-----+
|   1|              1|  109|
|   1|              2|  308|
|   2|              1|  191|
|   2|              2|  998|
|   3|              1|   60|
|   3|              2|  337|
|   4|              1|  146|
|   4|              2|  720|
|   5|              1|   26|
|   5|              2|  105|
+----+---------------+-----+

+-------+-----------------+-------------------+-----------------+------------------+
|summary|            pwgtp|    nycgov_pov_stat|    nycgov_income|  nycgov_threshold|
+-------+-----------------+-------------------+-----------------+------------------+
|  count|             3000|               3000|             3000|              3000|
|   mean|           125.32| 1.8226666666666667|76733.61543773311| 32511.14148799976|
| stddev|93.70736696514965|0.38201422379009375|65444.41895547774|12248.183053734736|
|    min|              100|                  1|            -1146|           16241.6

## 10. SAT: supresión y puntajes válidos

In [16]:
e=dfs['educacion']
sat=['num_of_sat_test_takers','sat_critical_reading_avg_score','sat_math_avg_score','sat_writing_avg_score']
for c in sat:
    e.groupBy(c).count().orderBy(F.desc('count')).show(5)
    e=e.withColumn(c+'_num',F.when(F.col(c).rlike(r'^\d+(\.\d+)?$'),F.col(c).cast('double')))
e.agg(*[F.count(c+'_num').alias(c+'_numerico') for c in sat]).show()
print('Los puntajes 2012 no son comparables directamente con SAT 2024 por cambio de escala, participación y cobertura estatal.')


+----------------------+-----+
|num_of_sat_test_takers|count|
+----------------------+-----+
|                     s|   57|
|                    54|   10|
|                    48|    8|
|                     9|    8|
|                    72|    8|
+----------------------+-----+
only showing top 5 rows
+------------------------------+-----+
|sat_critical_reading_avg_score|count|
+------------------------------+-----+
|                             s|   57|
|                           398|    8|
|                           384|    8|
|                           367|    8|
|                           370|    8|
+------------------------------+-----+
only showing top 5 rows
+------------------+-----+
|sat_math_avg_score|count|
+------------------+-----+
|                 s|   57|
|               385|    9|
|               391|    8|
|               364|    8|
|               381|    8|
+------------------+-----+
only showing top 5 rows
+---------------------+-----+
|sat_writing_avg_score|co

## Calidad y transformaciones
Reglas explícitas: conservar columnas de origen, normalizar fechas y códigos geográficos, marcar nulos, validar rangos, deduplicar solo por clave natural y separar agregaciones por unidad. No se imputan resultados sensibles. Las filas inválidas se etiquetan para revisión.

In [18]:
a=dfs['arrestos'].withColumn('fecha',F.to_date('arrest_date')).withColumn('boro',F.upper(F.trim('arrest_boro')))
a=a.withColumn('valida',F.col('arrest_key').isNotNull() & F.col('fecha').isNotNull() & F.col('boro').isin('B','K','M','Q','S'))
a=a.dropDuplicates(['arrest_key'])
v=dfs['vehiculos'].withColumn('fecha',F.to_date('crash_date')).withColumn('hora',F.to_timestamp('crash_time','HH:mm'))
v=v.withColumn('valida',F.col('unique_id').isNotNull() & F.col('collision_id').isNotNull() & F.col('fecha').isNotNull()).dropDuplicates(['unique_id'])
p=p.withColumn('boro_str',F.col('boro').cast('string')).withColumn('valida',F.col('serialno').isNotNull() & F.col('sporder').isNotNull() & (F.col('pwgtp')>0)).dropDuplicates(['serialno','sporder'])
e=e.withColumn('valida',F.col('dbn').isNotNull()).dropDuplicates(['dbn'])
for n,df in [('arrestos',a),('vehiculos',v),('pobreza',p),('educacion',e)]:
    print(n)
    df.groupBy('valida').count().show()


arrestos
+------+-----+
|valida|count|
+------+-----+
|  true|10000|
+------+-----+

vehiculos
+------+-----+
|valida|count|
+------+-----+
|  true| 3000|
+------+-----+

pobreza
+------+-----+
|valida|count|
+------+-----+
|  true| 3000|
+------+-----+

educacion
+------+-----+
|valida|count|
+------+-----+
|  true|  478|
+------+-----+



## Salidas analíticas propuestas
Las siguientes agregaciones sirven para exploración. No implican causalidad ni generan un ranking de intervención válido hasta verificar cobertura completa, denominadores poblacionales, calendario común y sesgos de registro.

In [20]:
a.filter('valida').groupBy('boro',F.date_format('fecha','yyyy-MM').alias('mes')).agg(F.countDistinct('arrest_key').alias('arrestos_extracto')).orderBy('boro','mes').show(50)
v.filter('valida').groupBy(F.date_format('fecha','yyyy-MM').alias('mes')).agg(F.countDistinct('collision_id').alias('colisiones_representadas_extracto')).orderBy('mes').show(50)


+----+-------+-----------------+
|boro|    mes|arrestos_extracto|
+----+-------+-----------------+
|   B|2026-01|             2335|
|   K|2026-01|             3092|
|   M|2026-01|             2199|
|   Q|2026-01|             2005|
|   S|2026-01|              369|
+----+-------+-----------------+

+-------+---------------------------------+
|    mes|colisiones_representadas_extracto|
+-------+---------------------------------+
|2012-07|                               58|
|2012-08|                               58|
|2012-09|                               66|
|2012-10|                               50|
|2012-11|                               52|
|2012-12|                               72|
|2013-01|                               51|
|2013-02|                               44|
|2013-03|                               55|
|2013-04|                               54|
|2013-05|                               56|
|2013-06|                               52|
|2013-07|                               54

## Criterios para fase siguiente
1. Descargar la población completa mediante paginación o exportación oficial y congelar corte.
2. Incorporar *Motor Vehicle Collisions – Crashes* para el indicador de siniestros, lesiones y geografía. El archivo obligatorio de vehículos permanece como detalle complementario.
3. Verificar diccionario de pobreza, ponderaciones y vigencia. SAT 2012 se presenta como antecedente; el informe estatal 2024 solo contextualiza.
4. Evaluar tendencias con periodos equivalentes y, si se propone intervención, medir antes y después con grupos de comparación y protección de privacidad.

**No ejecutado en un clúster institucional.** La configuración, capacidad y tiempos dependerán del entorno que facilite el curso.